# Yelp bias analysis — restaurant & cuisine preprocessing

Filter businesses to restaurants, assign ethnic cuisine labels, merge reviews in chunks, balance samples, clean text, and export baseline data.

**Inputs:** `yelp_academic_dataset_business.json` and `yelp_academic_dataset_review.json`. The user file is not needed for this baseline (no user-level features merged here).

In [ ]:
import gc
import os
import re
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------------------
# Paths: set YELP_DATA_DIR or edit DATA_DIR to point at your JSONL files.
# Outputs are written next to this notebook (project root).
# ---------------------------------------------------------------------------
DATA_DIR = Path(os.environ.get("YELP_DATA_DIR", "/Users/sherylcheng/Downloads/data"))
BUSINESS_FILE = DATA_DIR / "yelp_academic_dataset_business.json"
REVIEW_FILE = DATA_DIR / "yelp_academic_dataset_review.json"

PROJECT_ROOT = Path.cwd()
OUTPUT_CSV = PROJECT_ROOT / "processed_reviews_baseline.csv"
OUTPUT_SUMMARY = PROJECT_ROOT / "preprocessing_summary.txt"

REVIEW_CHUNKSIZE = 50_000
MAX_PER_CUISINE = 10_000
SAMPLE_RANDOM_STATE = 42

print(f"Data directory: {DATA_DIR}")
print(f"Business file exists: {BUSINESS_FILE.is_file()}")
print(f"Review file exists: {REVIEW_FILE.is_file()}")

## 1. Filter for restaurants only

Keep businesses whose `categories` string contains **Restaurants** (case-insensitive).

In [ ]:
print("[1/6] Loading business.json ...")
business_all = pd.read_json(BUSINESS_FILE, lines=True)
n_business_total = len(business_all)
print(f"  Total businesses loaded: {n_business_total:,}")

# Case-insensitive: categories string must contain "Restaurants"
cat_mask = business_all["categories"].str.contains(
    "Restaurants", case=False, na=False, regex=False
)
business_rest = business_all.loc[cat_mask].copy()
n_restaurants = len(business_rest)
print(f"  Businesses tagged as restaurants: {n_restaurants:,}")

del business_all
gc.collect()

## 2. Assign cuisine types

Match comma-separated `categories` against keywords; first cuisine in the fixed priority order wins.

In [ ]:
# Order matters: first matching cuisine in this list is assigned.
CUISINE_MAPPING = [
    ("Chinese", ["Chinese", "Cantonese", "Szechuan", "Dim Sum"]),
    ("Japanese", ["Japanese", "Sushi", "Ramen", "Izakaya"]),
    ("Korean", ["Korean"]),
    ("Thai", ["Thai"]),
    ("Vietnamese", ["Vietnamese", "Pho"]),
    ("Indian", ["Indian", "Pakistani", "Bangladeshi", "Himalayan"]),
    ("Mexican", ["Mexican", "Tex-Mex"]),
    ("Italian", ["Italian"]),
    (
        "American",
        ["American (Traditional)", "American (New)", "Burgers"],
    ),
    ("French", ["French"]),
]


def assign_cuisine_type(categories) -> str:
    """
    If any keyword appears in the categories string (case-insensitive),
    return the first matching cuisine in CUISINE_MAPPING order.
    Otherwise return 'Other'.
    """
    if pd.isna(categories) or not str(categories).strip():
        return "Other"
    text = str(categories).lower()
    for cuisine, keywords in CUISINE_MAPPING:
        for kw in keywords:
            if kw.lower() in text:
                return cuisine
    return "Other"


print("[2/6] Assigning cuisine_type ...")
business_rest["cuisine_type"] = business_rest["categories"].map(assign_cuisine_type)
print("  Distribution of cuisine_type (all restaurant rows):")
print(business_rest["cuisine_type"].value_counts())

n_with_cuisine = (business_rest["cuisine_type"] != "Other").sum()
print(f"\n  Restaurants with a mapped ethnic cuisine (not Other): {n_with_cuisine:,}")

## 3. Merge reviews with restaurant businesses (chunked)

Inner-merge each review chunk on `business_id`. Keep only rows where `cuisine_type != 'Other'`.

In [ ]:
# Slim lookup table for merges (frees column memory from full business_rest)
biz_cols = ["business_id", "cuisine_type", "name", "city", "state"]
biz_lookup = business_rest[biz_cols].drop_duplicates(subset=["business_id"])
del business_rest
gc.collect()
print(f"  Lookup businesses for merge: {len(biz_lookup):,}")

review_keep = ["review_id", "user_id", "business_id", "stars", "text", "date"]
merge_cols = review_keep + ["cuisine_type", "name", "city", "state"]

print("[3/6] Streaming review.json and merging ...")
total_reviews_before = 0
merged_chunks: list[pd.DataFrame] = []

for i, chunk in enumerate(
    pd.read_json(REVIEW_FILE, lines=True, chunksize=REVIEW_CHUNKSIZE)
):
    total_reviews_before += len(chunk)
    chunk = chunk[review_keep]
    m = chunk.merge(biz_lookup, on="business_id", how="inner")
    m = m.loc[m["cuisine_type"] != "Other", merge_cols]
    if len(m) > 0:
        merged_chunks.append(m)
    if (i + 1) % 20 == 0:
        print(
            f"  ... processed {(i + 1) * REVIEW_CHUNKSIZE:,}+ review rows "
            f"(file rows counted: {total_reviews_before:,})"
        )

print(f"\n  Total reviews in file (before merge filter): {total_reviews_before:,}")

reviews_merged = pd.concat(merged_chunks, ignore_index=True)
del merged_chunks
gc.collect()

total_after_merge = len(reviews_merged)
print(f"  Total reviews after merge (restaurant + mapped cuisine only): {total_after_merge:,}")

del biz_lookup
gc.collect()

## 4. Balance: up to 10,000 reviews per cuisine

Reproducible sampling with `random_state=42`.

In [ ]:
print("[4/6] Balancing per cuisine (max 10,000 each) ...")

balanced_parts: list[pd.DataFrame] = []
for cuisine, g in reviews_merged.groupby("cuisine_type"):
    n = min(MAX_PER_CUISINE, len(g))
    balanced_parts.append(g.sample(n=n, random_state=SAMPLE_RANDOM_STATE))

balanced = pd.concat(balanced_parts, ignore_index=True)
del balanced_parts

del reviews_merged
gc.collect()

print("  Final review counts per cuisine (after balancing):")
print(balanced["cuisine_type"].value_counts().sort_index())
print(f"\n  Total balanced reviews: {len(balanced):,}")

## 5. Basic text cleaning and length filter

Lowercase, strip URLs, normalize whitespace; drop reviews with fewer than 10 words.

In [ ]:
print("[5/6] Text cleaning ...")

# Match http(s) URLs and common www. spans (non-greedy until whitespace)
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
WHITESPACE_PATTERN = re.compile(r"\s+")


def clean_review_text(text: str) -> str:
    if pd.isna(text):
        return ""
    t = str(text).lower()
    t = URL_PATTERN.sub(" ", t)
    t = WHITESPACE_PATTERN.sub(" ", t).strip()
    return t


balanced["text_clean"] = balanced["text"].map(clean_review_text)
balanced["word_count"] = balanced["text_clean"].str.split().str.len().fillna(0).astype(int)

n_before_len = len(balanced)
removed_short = (balanced["word_count"] < 10).sum()
balanced = balanced.loc[balanced["word_count"] >= 10].copy()
balanced = balanced.drop(columns=["word_count"])

print(f"  Reviews before length filter: {n_before_len:,}")
print(f"  Reviews removed (< 10 words): {removed_short:,}")
print(f"  Reviews remaining: {len(balanced):,}")
print("\n  Distribution after length filter:")
print(balanced["cuisine_type"].value_counts().sort_index())

## 6. Save CSV and preprocessing summary

In [ ]:
print("[6/6] Saving outputs ...")

save_cols = [
    "review_id",
    "user_id",
    "business_id",
    "stars",
    "text",
    "text_clean",
    "date",
    "cuisine_type",
    "name",
    "city",
    "state",
]
balanced[save_cols].to_csv(OUTPUT_CSV, index=False)
print(f"  Wrote {OUTPUT_CSV}")

sample_lines = []
sample = balanced.head(5)[["text_clean", "stars", "cuisine_type"]]
for _, row in sample.iterrows():
    tc = row["text_clean"]
    if len(tc) > 400:
        tc = tc[:400] + "..."
    sample_lines.append(
        f"  - [{row['cuisine_type']}] stars={row['stars']}: {tc!r}"
    )
sample_block = "\n".join(sample_lines)

summary_text = f"""Yelp bias analysis — preprocessing summary
============================================================

BUSINESSES
  Total businesses (file): {n_business_total:,}
  Restaurants (categories contain 'Restaurants'): {n_restaurants:,}
  Restaurants with mapped ethnic cuisine (not Other): {n_with_cuisine:,}

REVIEWS
  Total reviews in review.json: {total_reviews_before:,}
  After merge (restaurant + mapped cuisine, excluding Other): {total_after_merge:,}
  After balancing (max {MAX_PER_CUISINE:,} per cuisine): {n_before_len:,}
  Removed by length filter (< 10 words): {removed_short:,}
  Final rows written to CSV: {len(balanced):,}

FINAL DISTRIBUTION (cuisine_type)
{balanced['cuisine_type'].value_counts().sort_index().to_string()}

SAMPLE REVIEWS (text_clean, stars, cuisine_type)
{sample_block}
"""

OUTPUT_SUMMARY.write_text(summary_text, encoding="utf-8")
print(f"  Wrote {OUTPUT_SUMMARY}")
print("\nDone.")